In [47]:
import os
import pandas as pd
import numpy as np
import pickle
import yaml

import warnings

from statsmodels.tsa.api import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.statespace.varmax import VARMAX
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

In [48]:
warnings.filterwarnings("ignore")

In [49]:
df = pd.read_csv("../../data/pre_data.csv")
df["Ngày"] = pd.to_datetime(df["Ngày"], format="%Y-%m-%d")

In [50]:
df.head()

,Tên_mặt_hàng,Thị_trường,Loại_giá,Nguồn,Ngày,Giá
0,OM 5451,Cần Thơ,Thương lái thu mua,CTV địa phương,2025-05-19,6000.0
1,OM 5451,Sóc Trăng,Thương lái thu mua,CTV địa phương,2025-05-19,7125.0
2,Gạo NL 25% tấm,Kiên Giang,Thương lái thu mua,CTV địa phương,2025-05-16,8460.0
3,Gạo XK 5% tấm,Kiên Giang,Thương lái thu mua,CTV địa phương,2025-05-16,10070.0
4,OM 5451,Kiên Giang,Thương lái thu mua,CTV địa phương,2025-05-16,5900.0


In [51]:
df[df["Tên_mặt_hàng"] == "Gạo NL 25% tấm"]

,Tên_mặt_hàng,Thị_trường,Loại_giá,Nguồn,Ngày,Giá
2,Gạo NL 25% tấm,Kiên Giang,Thương lái thu mua,CTV địa phương,2025-05-16,8460.0
13,Gạo NL 25% tấm,Kiên Giang,Thương lái thu mua,CTV địa phương,2025-05-09,8520.0
27,Gạo NL 25% tấm,Kiên Giang,Thương lái thu mua,CTV địa phương,2025-04-25,8510.0
116,Gạo NL 25% tấm,Kiên Giang,Thu mua,Giồng Riềng,2024-12-20,9930.0
121,Gạo NL 25% tấm,Kiên Giang,Thu mua,Giồng Riềng,2024-12-18,9980.0
...,...,...,...,...,...,...
1649,Gạo NL 25% tấm,Kiên Giang,Khác,CTV địa phương,2023-01-12,9600.0
1659,Gạo NL 25% tấm,Kiên Giang,Thương lái thu mua,CTV địa phương,2023-01-11,9800.0
1671,Gạo NL 25% tấm,Kiên Giang,Thương lái thu mua,CTV địa phương,2023-01-06,9600.0
1686,Gạo NL 25% tấm,Kiên Giang,Khác,CTV địa phương,2023-01-05,9600.0


In [52]:
for col in ["Thị_trường", "Loại_giá", "Nguồn"]:
    lbl_encoder = LabelEncoder()
    df[col] = lbl_encoder.fit_transform(df[col])
    print(lbl_encoder.classes_)

    with open(f"../train/lbl_scaler/{col}.pkl", "wb") as file:
        pickle.dump(lbl_encoder, file)

['An Giang' 'Bạc Liêu' 'Bến Tre' 'Cà Mau' 'Cần Thơ' 'Hà Nội' 'Hậu Giang'
 'Kiên Giang' 'Sóc Trăng' 'Sơn La' 'Thái Bình' 'Tiền Giang' 'Trà Vinh'
 'Vĩnh Long' 'Đồng Tháp']
['Bán buôn' 'Bán lẻ' 'Khác' 'Thu mua' 'Thu mua tại vườn'
 'Thương lái thu mua' 'Tại chợ' 'Vựa thu mua' 'Đại lý thu mua']
['CTV địa phương' 'Giồng Riềng' 'Long Xuyên' 'Tỉnh An Giang']


In [53]:
for col in ["Thị_trường", "Loại_giá", "Nguồn"]:
    scaler = MinMaxScaler()  
    df[col] = scaler.fit_transform(df[[col]])

    with open(f"../train/mm_scaler/{col}.pkl", "wb") as file:
        pickle.dump(scaler, file)

In [54]:
def get_mm_dict():
    cols = ["Thị_trường", "Loại_giá", "Nguồn"]
    result = dict()

    for col in cols:
        with open(f"../train/mm_scaler/{col}.pkl", "rb") as file:
            mm_scaler = pickle.load(file)
            result.update({col: mm_scaler})
    
    return result

def get_lbl_dict():
    cols = ["Thị_trường", "Loại_giá", "Nguồn"]
    result = dict()

    for col in cols:
        with open(f"../train/lbl_scaler/{col}.pkl", "rb") as file:
            mm_scaler = pickle.load(file)
            result.update({col: mm_scaler})
    
    return result

In [55]:
mm_dict = get_mm_dict()
mm_dict

{'Thị_trường': MinMaxScaler(),
 'Loại_giá': MinMaxScaler(),
 'Nguồn': MinMaxScaler()}

In [56]:
lbl_dict = get_lbl_dict()
lbl_dict

{'Thị_trường': LabelEncoder(),
 'Loại_giá': LabelEncoder(),
 'Nguồn': LabelEncoder()}

In [57]:
df.head()

,Tên_mặt_hàng,Thị_trường,Loại_giá,Nguồn,Ngày,Giá
0,OM 5451,0.285714,0.625,0.0,2025-05-19,6000.0
1,OM 5451,0.571429,0.625,0.0,2025-05-19,7125.0
2,Gạo NL 25% tấm,0.500000,0.625,0.0,2025-05-16,8460.0
3,Gạo XK 5% tấm,0.500000,0.625,0.0,2025-05-16,10070.0
4,OM 5451,0.500000,0.625,0.0,2025-05-16,5900.0


In [58]:
df.head()

,Tên_mặt_hàng,Thị_trường,Loại_giá,Nguồn,Ngày,Giá
0,OM 5451,0.285714,0.625,0.0,2025-05-19,6000.0
1,OM 5451,0.571429,0.625,0.0,2025-05-19,7125.0
2,Gạo NL 25% tấm,0.500000,0.625,0.0,2025-05-16,8460.0
3,Gạo XK 5% tấm,0.500000,0.625,0.0,2025-05-16,10070.0
4,OM 5451,0.500000,0.625,0.0,2025-05-16,5900.0


In [ ]:
items = df["Tên_mặt_hàng"].unique()

results = {}

# Đọc dữ liệu
df["Ngày"] = pd.to_datetime(df["Ngày"], dayfirst=True)
items = df["Tên_mặt_hàng"].unique().tolist()

with open("../../data/metadata/u_items.yaml", "w", encoding="utf-8") as file:
    yaml.dump(items, file, encoding="utf-8", allow_unicode=True, sort_keys=False)

results = {}

for idx, item in enumerate(items):
    print(item)
    item_df = df[df["Tên_mặt_hàng"] == item].sort_values("Ngày")

    # Đặt chỉ số thời gian
    item_df = item_df.set_index("Ngày")

    # Target variable
    y = item_df["Giá"]

    # Encode exogenous variables
    exog = item_df[["Thị_trường", "Loại_giá", "Nguồn"]]

    # Huấn luyện mô hình SARIMAX (ARIMA với exog)
    try:
        model = SARIMAX(y, exog=exog, order=(1,1,1))
        result = model.fit(disp=False)
        results[item] = result

        with open(f"../../models/{idx}.pkl", "wb") as file:
            pickle.dump(result, file)
    except Exception as e:
        print(f"Lỗi khi huấn luyện {item}: {e}")

OM 5451
Gạo NL 25% tấm
Gạo XK 5% tấm
ST24
Lúa IR 50404
Lúa Jasmine
Gạo thơm Đài Loan (trong)
Gạo NL 15%
Gạo Sóc thường
Lúa OM 5451
Gạo Hương Lài
Gạo Jasmine
Lúa IR Jasmine (lúa tươi)
Gạo IR50404
Lúa OM 4218
Gạo Bắc thơm
Lúa Bắc thơm
Lúa Khang dân
Lúa ST24
Lúa ST 24 (lúa tươi)
Lúa thường IR 50404 (khô)
Lúa thường IR 50404 (tươi)
Lúa ST20 (lúa tươi)
Lúa Q5 (lúa tươi)
Lúa T10 (lúa tươi)
Lúa Thiên Ưu (lúa tươi)
Bắp cải
Bưởi da xanh
Cà chua
Cà rốt
Chanh
Đậu cove
Dưa chuột/dưa chuột
Khoai tây
Mận hậu
Mít thái
Mồng tơi
Mướp đắng
Mướp hương
Quả bí đỏ
Quả bí xanh
Rau cải
Rau ngót
Su su quả
Cam sành
Măng cụt
Mướp
Rau cải mơ
Rau cải ngọt
Rau muống
Bưởi long Núm
Bưởi năm roi
Chôm chôm Java
Chuối sim
Đu đủ
Hồng xiêm
Sầu Riêng Ri 6 (loại đẹp)
Sầu Riêng Ri 6 (loại xô)
Rau cần
Vú sữa
Na thái
Na
Xoài Đài Loan
Nhãn tiêu quế
Dưa chuột/dưa chuội
Cam xoàn
Quýt đường
Xoài cát Hòa Lộc
Bưởi năm roi (loại 1)
Cam sành (loại 1)
Cam sành (loại 2)
Chôm chôm đường
Sầu riêng cơm vàng hạt lép
Trái dứa/thơm (nhỏ)
Trái

# Test

In [60]:
groups = df.groupby(["Tên_mặt_hàng", "Thị_trường", "Loại_giá", "Nguồn"])
results = []

for keys, group_df in groups:
    group_df = group_df.sort_values("Ngày")
    
    if len(group_df) < 10:
        continue  # Skip if not enough data

    y = group_df["Giá"]
    exog = group_df[["Tên_mặt_hàng", "Thị_trường", "Loại_giá", "Nguồn"]]

    try:
        model = SARIMAX(y, exog=exog, order=(1,1,1), seasonal_order=(0,0,0,0))
        model_fit = model.fit(disp=False)
        with open("./sarimax.pkl", "wb") as file:
            pickle.dump(model_fit, file)
        
        # Forecast using the last row of exog
        forecast = model_fit.forecast(steps=1, exog=exog.tail(1))
        results.append((keys, forecast.iloc[0]))

    except Exception as e:
        print(f"Skip group {keys} due to error: {e}")


Skip group ('Bông cải Xanh', np.float64(0.21428571428571427), np.float64(0.75), np.float64(0.0)) due to error: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).
Skip group ('Bưởi da xanh', np.float64(0.3571428571428571), np.float64(0.125), np.float64(0.0)) due to error: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).
Skip group ('Bưởi da xanh', np.float64(0.6428571428571428), np.float64(0.125), np.float64(0.0)) due to error: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).
Skip group ('Bưởi da xanh', np.float64(0.7857142857142857), np.float64(0.5), np.float64(0.0)) due to error: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).
Skip group ('Bưởi da xanh', np.float64(0.7857142857142857), np.float64(0.625), np.float64(0.0)) due to error: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).
Skip group ('Bưởi da xanh (loại 1)',

In [61]:

groups = df.groupby(["Tên_mặt_hàng", "Thị_trường", "Loại_giá", "Nguồn"])
results = []

for keys, group_df in groups:
    group_df = group_df.sort_values("Ngày")
    
    if len(group_df) < 10:
        continue  # Skip if not enough data

    y = group_df["Giá"]
    exog = group_df[["Tên_mặt_hàng", "Thị_trường", "Loại_giá", "Nguồn"]]

    try:
        model = ARIMA(y, exog=exog, order=(1,1,1))
        model_fit = model.fit()
        with open("./arima.pkl", "wb") as file:
            pickle.dump(model_fit, file)
        
        # Forecast using the last row of exog
        forecast = model_fit.forecast(steps=10, exog=exog.tail(1))
        results.append((keys, forecast.iloc[0]))

    except Exception as e:
        print(f"Skip group {keys} due to error: {e}")


Skip group ('Bông cải Xanh', np.float64(0.21428571428571427), np.float64(0.75), np.float64(0.0)) due to error: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).
Skip group ('Bưởi da xanh', np.float64(0.3571428571428571), np.float64(0.125), np.float64(0.0)) due to error: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).
Skip group ('Bưởi da xanh', np.float64(0.6428571428571428), np.float64(0.125), np.float64(0.0)) due to error: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).
Skip group ('Bưởi da xanh', np.float64(0.7857142857142857), np.float64(0.5), np.float64(0.0)) due to error: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).
Skip group ('Bưởi da xanh', np.float64(0.7857142857142857), np.float64(0.625), np.float64(0.0)) due to error: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).
Skip group ('Bưởi da xanh (loại 1)',